In [2]:
import sys
print(sys.executable)

c:\Users\BrunettiLeonardo\nfl-agent\.venv\Scripts\python.exe


In [2]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd

csv_path = "../data/games.csv"
games = pd.read_csv(csv_path)

games.head()

,gameId,season,week,gameDate,gameTimeEastern,homeTeamAbbr,visitorTeamAbbr,homeFinalScore,visitorFinalScore
0,2022090800,2022,1,9/8/2022,20:20:00,LA,BUF,10,31
1,2022091100,2022,1,9/11/2022,13:00:00,ATL,NO,26,27
2,2022091101,2022,1,9/11/2022,13:00:00,CAR,CLE,24,26
3,2022091102,2022,1,9/11/2022,13:00:00,CHI,SF,19,10
4,2022091103,2022,1,9/11/2022,13:00:00,CIN,PIT,20,23


# NFL Game Results Agent

In this notebook, we will build the first version of an NFL data-analysis agent.

The starting dataset contains NFL games with:
- season
- week
- game date and time
- home team
- visiting team
- final score

Before connecting an LLM, we first prepare the dataset so it becomes easier to query.

In [2]:
games.info()

<class 'pandas.DataFrame'>
RangeIndex: 136 entries, 0 to 135
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   gameId             136 non-null    int64
 1   season             136 non-null    int64
 2   week               136 non-null    int64
 3   gameDate           136 non-null    str  
 4   gameTimeEastern    136 non-null    str  
 5   homeTeamAbbr       136 non-null    str  
 6   visitorTeamAbbr    136 non-null    str  
 7   homeFinalScore     136 non-null    int64
 8   visitorFinalScore  136 non-null    int64
dtypes: int64(5), str(4)
memory usage: 9.7 KB


## Data Preparation

The raw dataset already contains the final score for each game, but we can make it more useful by adding derived columns.

We will add:

- `homeWin`: whether the home team won
- `winner`: the team that won the game
- `loser`: the team that lost the game
- `totalPoints`: total points scored by both teams
- `scoreMargin`: absolute difference between the two final scores

In [6]:
games["homeWin"] = games["homeFinalScore"] > games["visitorFinalScore"]

games["winner"] = games.apply(
    lambda row: row["homeTeamAbbr"] if row["homeWin"] else row["visitorTeamAbbr"],
    axis = 1
)

games["loser"] = games.apply(
    lambda row: row["visitorTeamAbbr"] if row ["homeWin"] else row["homeTeamAbbr"],
    axis = 1
)

games["totalPoints"] = games["homeFinalScore"] + games["visitorFinalScore"]
games["scoreMargin"] = abs(games["homeFinalScore"] - games["visitorFinalScore"])

games.head()

,gameId,season,week,gameDate,gameTimeEastern,homeTeamAbbr,visitorTeamAbbr,homeFinalScore,visitorFinalScore,homeWin,winner,loser,totalPoints,scoreMargin
0,2022090800,2022,1,9/8/2022,20:20:00,LA,BUF,10,31,False,BUF,LA,41,21
1,2022091100,2022,1,9/11/2022,13:00:00,ATL,NO,26,27,False,NO,ATL,53,1
2,2022091101,2022,1,9/11/2022,13:00:00,CAR,CLE,24,26,False,CLE,CAR,50,2
3,2022091102,2022,1,9/11/2022,13:00:00,CHI,SF,19,10,True,CHI,SF,29,9
4,2022091103,2022,1,9/11/2022,13:00:00,CIN,PIT,20,23,False,PIT,CIN,43,3


## Quick Data Checks

Before building the agent, we run a few simple checks to understand what is inside the dataset.

This helps us know what kinds of questions the agent will be able to answer.

In [8]:
print("Rows: ", len(games))
print("Columns: ", len(games.columns))
print("Seasons: ", sorted(games["season"].unique()))
print("Weeks: ", games["week"].min(), "to ", games["week"].max())
print("Teams :", len(set(games["homeTeamAbbr"]) | set(games["visitorTeamAbbr"])))

Rows:  136
Columns:  14
Seasons:  [np.int64(2022)]
Weeks:  1 to  9
Teams : 32


## Basic NFL Questions with Pandas

Before using an LLM, we answer a few questions directly with Pandas.

This is useful because:
- it helps us verify the dataset
- it gives us examples of the kind of logic the agent will later automate
- it makes debugging much easier

In [17]:
print("Which games had the largest score margin?")
games.sort_values("scoreMargin", ascending=False).head(10)[
    [
        "week","homeTeamAbbr", "visitorTeamAbbr", "homeFinalScore", "visitorFinalScore", "winner", "scoreMargin"
    ]
]

Which games had the largest score margin?


,week,homeTeamAbbr,visitorTeamAbbr,homeFinalScore,visitorFinalScore,winner,scoreMargin
66,5,BUF,PIT,38,3,BUF,35
30,2,BUF,TEN,41,7,BUF,34
70,5,NE,DET,29,0,NE,29
42,3,LAC,JAX,10,38,JAX,28
114,8,NO,LV,24,0,NO,24
20,2,JAX,IND,24,0,JAX,24
10,1,ARI,KC,21,44,KC,23
72,5,NYJ,MIA,40,17,NYJ,23
82,6,CLE,NE,15,38,NE,23
129,9,NE,IND,26,3,NE,23


## Team Records

Now we calculate each team's win-loss record using the available games.

Because the dataset only covers weeks 1-9 of the 2022 season, these are not full-season records.

In [25]:
wins = games["winner"]. value_counts()
losses = games["loser"].value_counts()

team_records = pd.DataFrame({
    "wins": wins,
    "losses": losses
}).fillna(0)

team_records["wins"]= team_records ["wins"].astype(int)
team_records["losses"] = team_records["losses"].astype(int)
team_records["gamesPlayed"] = team_records["wins"] + team_records["losses"]
team_records["winPct"] = team_records["wins"]/team_records["gamesPlayed"]
team_records.sort_values(["wins", "winPct"], ascending = False).head(10)

,wins,losses,gamesPlayed,winPct
PHI,8,0,8,1.000000
MIN,7,1,8,0.875000
BUF,6,2,8,0.750000
DAL,6,2,8,0.750000
KC,6,2,8,0.750000
NYG,6,2,8,0.750000
BAL,6,3,9,0.666667
MIA,6,3,9,0.666667
NYJ,6,3,9,0.666667
SEA,6,3,9,0.666667


## Highest Scoring Games

Next, we identify the games with the highest combined score.

In [26]:
games.sort_values("totalPoints", ascending=False).head(10)[
    ["week", "homeTeamAbbr", "visitorTeamAbbr","homeFinalScore", "visitorFinalScore", "winner", "totalPoints"]
]

,week,homeTeamAbbr,visitorTeamAbbr,homeFinalScore,visitorFinalScore,winner,totalPoints
53,4,DET,SEA,45,48,SEA,93
17,2,BAL,MIA,38,42,MIA,80
111,8,DAL,CHI,49,29,DAL,78
94,7,ARI,NO,42,34,ARI,76
5,1,DET,PHI,35,38,PHI,73
62,4,TB,KC,31,41,KC,72
110,8,ATL,CAR,37,34,ATL,71
71,5,NO,SEA,39,32,NO,71
125,9,CHI,MIA,32,35,MIA,67
105,7,SF,KC,23,44,KC,67


## Preparing the LLM Connection

The next step is to connect the notebook to an LLM through Amazon Bedrock.

We will use Bedrock as the model platform, and LangChain as the Python interface.

Before running any LLM calls, we need:
- AWS credentials configured on this machine
- Bedrock model access enabled in the AWS account
- the correct AWS region
- the correct Bedrock model ID

## Choosing the Bedrock Model

Because Anthropic models are not available for our use case, we will use an Amazon model through Bedrock.

For the first version, we will use:

`amazon.nova-pro-v1:0`

This is a good default model for reasoning over structured data and natural-language questions.

Later, we can compare it with cheaper or faster models such as:

- `amazon.nova-lite-v1:0`
- `meta.llama3-3-70b-instruct-v1:0`
- `openai.gpt-oss-120b-1:0`

In [27]:
%pip install langchain langchain-aws langchain-experimental langgraph boto3 python-dotenv


Note: you may need to restart the kernel to use updated packages.


In [31]:
from langchain_aws import ChatBedrockConverse

llm = ChatBedrockConverse(
    model='amazon.nova-lite-v1:0',
    region_name = "us-east-1",
    temperature=0
)

response = llm.invoke("Reply with one short sentence, like <Bedrock connection works>")
print(response.content)

<Bedrock connection works>


## Creating the NFL Games Data Agent

Now we create the first agent.

This agent receives a natural-language question, writes Python/Pandas logic internally, and uses the `games` dataframe to answer.

For now, the agent only knows the data inside `games.csv`.

In [34]:
%pip install tabulate

  Using cached tabulate-0.10.0-py3-none-any.whl.metadata (40 kB)
Using cached tabulate-0.10.0-py3-none-any.whl (39 kB)
Note: you may need to restart the kernel to use updated packages.


In [39]:
from langchain_experimental.agents import create_pandas_dataframe_agent

games_agent = create_pandas_dataframe_agent(
    llm,
    games,
    agent_type="tool-calling",
    verbose=True,
    allow_dangerous_code=True,
    agent_executor_kwargs={"handle_parsing_errors": True},
)

In [40]:
result = games_agent.invoke({
    "input": "Which game had the largest score margin? Include the week, teams, final score, winner and margin." 
})
print(result["output"])



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "df.loc[df['scoreMargin'].idxmax()]"}`
responded: [{'type': 'text', 'text': "<thinking> To find the game with the largest score margin, I need to identify the row in the dataframe with the highest value in the 'scoreMargin' column. I can do this by using the 'idxmax' function on the 'scoreMargin' column. Once I have the index of the row with the largest score margin, I can use it to retrieve the relevant information from the dataframe. </thinking>\n", 'index': 0}, {'type': 'tool_use', 'name': 'python_repl_ast', 'id': 'tooluse_QQiVrHEuwAUO7lK76pLw83', 'index': 1, 'input': '{"query":"df.loc[df[\'scoreMargin\'].idxmax()]"}'}]

gameId               2022100901
season                     2022
week                          5
gameDate              10/9/2022
gameTimeEastern        13:00:00
homeTeamAbbr                BUF
visitorTeamAbbr             PIT
homeFinalScore               38
visitorFinalScore          

## First Agent Test

The first successful agent question was:

> Which game had the largest score margin?

The agent correctly used the dataframe and found:

- Week 5
- Buffalo Bills vs Pittsburgh Steelers
- Buffalo won 38-3
- Score margin: 35

This confirms that the Bedrock model can call a Python/Pandas tool and answer questions from the NFL dataset.

In [41]:
result = games_agent.invoke({
    "input": "Using only the available games in the dataframe, what was each team's win-loss record? Show the top 10 teams by wins."
})

print(result["output"])



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "wins = df.groupby('winner')['winner'].count().reset_index(name='wins')"}`
responded: [{'type': 'text', 'text': "<thinking> To determine each team's win-loss record, I need to count the number of wins for each team. I can do this by grouping the dataframe by the 'winner' column and counting the number of occurrences for each team. Then, I can do the same for the 'loser' column to get the number of losses. After that, I can merge these two results to get the win-loss record for each team. Finally, I can sort the teams by their number of wins to get the top 10 teams by wins. </thinking>\n", 'index': 0}, {'type': 'tool_use', 'name': 'python_repl_ast', 'id': 'tooluse_P4GfDcXA8GdcK2c3rqiahR', 'index': 1, 'input': '{"query":"wins = df.groupby(\'winner\')[\'winner\'].count().reset_index(name=\'wins\')"}'}, {'type': 'tool_use', 'name': 'python_repl_ast', 'id': 'tooluse_C95teoDAZ2hsTanFGB1qTe', 'index': 2, 'inp

## Safer Deterministic Tools

The Pandas agent can answer questions, but it may sometimes write incorrect Python code.

For common football questions, we can create deterministic Python functions.

This gives us:
- more reliable answers
- easier debugging
- less dependence on the model's ability to write code
- a better foundation for a production agent

In [50]:
def get_team_records(games_df):
    wins = games_df["winner"].value_counts()
    losses = games_df["loser"].value_counts()

    records = pd.DataFrame({
        "wins":wins,
        "losses":losses
    }).fillna(0)

    records["wins"] = records["wins"].astype(int)
    records["losses"] = records["losses"].astype(int)
    records["gamesPlayed"] = records["wins"] + records["losses"]
    records["winPct"] = records["wins"] / records["gamesPlayed"]
    records["winPct"] = records["winPct"].round(3)

    records = records.reset_index().rename(columns={"index": "team"})
    records = records.sort_values(
        ["wins", "winPct", "team"],
        ascending = [False, False, True]
    )

    return records

In [51]:
team_records = get_team_records(games)
team_records.head(10)

,team,wins,losses,gamesPlayed,winPct
25,PHI,8,0,8,1.000
20,MIN,7,1,8,0.875
3,BUF,6,2,8,0.750
8,DAL,6,2,8,0.750
15,KC,6,2,8,0.750
23,NYG,6,2,8,0.750
2,BAL,6,3,9,0.667
19,MIA,6,3,9,0.667
24,NYJ,6,3,9,0.667
27,SEA,6,3,9,0.667


## More Football Analysis Tools

Now we add a few reusable functions for common NFL questions.

These functions are deterministic: they do not depend on the LLM writing correct Pandas code.

In [61]:
def get_largest_score_margins(games_df, n=10):
    columns=[
        'week',
        'gameDate',
        'homeTeamAbbr',
        'visitorTeamAbbr',
        'homeFinalScore',
        'visitorFinalScore',
        'winner',
        'scoreMargin'
    ]
    return games_df.sort_values("scoreMargin", ascending = False)[columns].head(n)

def get_higest_scoring_games(games_df, n=10):
    columns=[
        "week",
        "gameDate",
        "homeTeamAbbr",
        "visitorTeamAbbr",
        "homeFinalScore",
        "visitorFinalScore",
        "winner",
        "totalPoints" 
    ]
    return games_df.sort_values("totalPoints", ascending=False)[columns].head(n)

def get_team_games(games_df, team):
    team = team.upper()
    team_games = games_df[
        (games_df['homeTeamAbbr'] == team)  | 
        (games_df['visitorTeamAbbr'] == team)
    ]

    columns = [
            "week",
            "gameDate",
            "homeTeamAbbr",
            "visitorTeamAbbr",
            "homeFinalScore",
            "visitorFinalScore",
            "winner",
            "scoreMargin",
            "totalPoints"
        ]

    return team_games.sort_values("week")[columns]


In [62]:
get_largest_score_margins(games, n=5)

,week,gameDate,homeTeamAbbr,visitorTeamAbbr,homeFinalScore,visitorFinalScore,winner,scoreMargin
66,5,10/9/2022,BUF,PIT,38,3,BUF,35
30,2,9/19/2022,BUF,TEN,41,7,BUF,34
70,5,10/9/2022,NE,DET,29,0,NE,29
42,3,9/25/2022,LAC,JAX,10,38,JAX,28
114,8,10/30/2022,NO,LV,24,0,NO,24


In [63]:
get_highest_scoring_games(games, n=5)

,week,gameDate,homeTeamAbbr,visitorTeamAbbr,homeFinalScore,visitorFinalScore,winner,totalPoints
53,4,10/2/2022,DET,SEA,45,48,SEA,93
17,2,9/18/2022,BAL,MIA,38,42,MIA,80
111,8,10/30/2022,DAL,CHI,49,29,DAL,78
94,7,10/20/2022,ARI,NO,42,34,ARI,76
5,1,9/11/2022,DET,PHI,35,38,PHI,73


In [65]:
get_team_games(games, "NE")

,week,gameDate,homeTeamAbbr,visitorTeamAbbr,homeFinalScore,visitorFinalScore,winner,scoreMargin,totalPoints
7,1,9/11/2022,MIA,NE,20,7,MIA,13,27
23,2,9/18/2022,PIT,NE,14,17,NE,3,31
38,3,9/25/2022,NE,BAL,26,37,BAL,11,63
60,4,10/2/2022,GB,NE,27,24,GB,3,51
70,5,10/9/2022,NE,DET,29,0,NE,29,29
82,6,10/16/2022,CLE,NE,15,38,NE,23,53
107,7,10/24/2022,NE,CHI,14,33,CHI,19,47
115,8,10/30/2022,NYJ,NE,17,22,NE,5,39
129,9,11/6/2022,NE,IND,26,3,NE,23,29


## Turning Functions into Agent Tools

Now we expose our deterministic Python functions as LangChain tools.

Instead of letting the LLM write arbitrary Pandas code, we give it a small set of reliable tools.

The model's job becomes:
1. understand the user's question
2. choose the right tool
3. explain the result clearly

In [66]:
from langchain_core.tools import tool

@tool
def team_records_tool() -> str:
    '''return the win-loss for all teams, sorted by wins and win percentage'''
    records = get_team_records(games)
    return records.to_markdown(index=False)

@tool
def largest_score_margins_tool(n: int = 10) -> str:
    """return the games with the largest score margins"""
    margins = get_largest_score_margins(games, n=n)
    return margins.to_markdown(index=False)

@tool
def highest_scoring_games_tool(n: int=10) -> str:
    """Return the games with the highest combined total points."""
    high_scores = get_highest_scoring_games(games, n=n)
    return high_scores.to_markdown(index=False)


@tool
def team_games_tool(team: str) -> str:
    """return all available games for a specific NFL team abbraviatio, such as BUF, PHI..."""
    team_games = get_team_games(games, team)

    if team_games.empty:
        return f"No games found for team abbreviaiton: {team.upper()}"
    return team_games.to_markdown(index=False)
                                   



In [87]:
tools = [ team_records_tool, largest_score_margins_tool, highest_scoring_games_tool, team_games_tool]
for tool_item in tool:
    print(tool_item.name, ' - ', tool_item.description)

team_records_tool  -  return the win-loss for all teams, sorted by wins and win percentage
largest_score_margins_tool  -  return the games with the largest score margins
highest_scoring_games_tool  -  Return the games with the highest combined total points.
team_games_tool  -  return all available games for a specific NFL team abbraviatio, such as BUF, PHI...


## Binding Tools to the LLM

Now we bind the deterministic tools to the Bedrock model.

This means the model can decide when to call one of our Python tools instead of trying to calculate everything by itself.

In [88]:
llm_with_tools = llm.bind_tools(tool)

In [89]:
response = llm_with_tools.invoke(
    "Which tool should you use to show the top team records?"
)
response

AIMessage(content=[{'type': 'text', 'text': "<thinking> The User wants to see the top team records. The tool that can provide this information is the 'team_records_tool', which returns the win-loss records for all teams, sorted by wins and win percentage. </thinking>\n"}, {'type': 'tool_use', 'name': 'team_records_tool', 'input': {}, 'id': 'tooluse_sm3XIIpfpx4NrvFu7zowld'}], additional_kwargs={}, response_metadata={'ResponseMetadata': {'RequestId': '1bad3b28-0c86-47e7-aa08-a9c65abca55c', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Tue, 19 May 2026 13:31:37 GMT', 'content-type': 'application/json', 'content-length': '525', 'connection': 'keep-alive', 'x-amzn-requestid': '1bad3b28-0c86-47e7-aa08-a9c65abca55c'}, 'RetryAttempts': 0}, 'stopReason': 'tool_use', 'metrics': {'latencyMs': [633]}, 'model_provider': 'bedrock_converse', 'model_name': 'amazon.nova-lite-v1:0'}, id='lc_run--019e406f-4c80-7fa0-b77c-93bcb7559daf-0', tool_calls=[{'name': 'team_records_tool', 'args': {}, 'id': 'toolu

In [90]:
team_records_tool

StructuredTool(name='team_records_tool', description='return the win-loss for all teams, sorted by wins and win percentage', args_schema=<class 'langchain_core.utils.pydantic.team_records_tool'>, func=<function team_records_tool at 0x000002046FA2F7E0>)

## Building a Tool-Calling Graph with LangGraph

Now we build a small LangGraph workflow.

The graph will:
1. send the user question to the model
2. let the model choose a tool
3. execute the selected tool
4. send the tool result back to the model
5. return a final natural-language answer

In [94]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

In [99]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

In [100]:
tool_node = ToolNode(tools)

def call_model(state: AgentState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

In [101]:
graph_builder = StateGraph(Agentstate)
graph_builder.add_node("model", call_model)
graph_builder.add_node("tools", tool_node)
graph_builder.add_edge(START, "model")
graph_builder.add_conditional_edges(
    "model",
    tools_condition
)
graph_builder.add_edge("tools", "model")

nfl_agent_graph = graph_builder.compile()

In [104]:
response = nfl_agent_graph.invoke({
    "messages": [
        ("user", "show me the top 10 teams by record in the available games. Return the answer as a numbered list")
    ]
})
print(response["messages"][-1].content)

Here are the top 10 teams by record in the available games:

1. **PHI** - 8 wins, 0 losses, 8 games played, win percentage: 1.000
2. **MIN** - 7 wins, 1 loss, 8 games played, win percentage: 0.875
3. **BUF** - 6 wins, 2 losses, 8 games played, win percentage: 0.750
4. **DAL** - 6 wins, 2 losses, 8 games played, win percentage: 0.750
5. **KC** - 6 wins, 2 losses, 8 games played, win percentage: 0.750
6. **NYG** - 6 wins, 2 losses, 8 games played, win percentage: 0.750
7. **BAL** - 6 wins, 3 losses, 9 games played, win percentage: 0.667
8. **MIA** - 6 wins, 3 losses, 9 games played, win percentage: 0.667
9. **NYJ** - 6 wins, 3 losses, 9 games played, win percentage: 0.667
10. **SEA** - 6 wins, 3 losses, 9 games played, win percentage: 0.667

<thinking> The tool has provided the top 10 teams by record, and I have formatted the results as a numbered list. </thinking>


## First LangGraph Agent Result

The LangGraph agent is now working.

Instead of asking the model to write arbitrary Pandas code, we gave it reliable tools:

- `team_records_tool`
- `largest_score_margins_tool`
- `highest_scoring_games_tool`
- `team_games_tool`

The model chooses the right tool, LangGraph executes it, and the model turns the result into a natural-language answer.

This is a safer and more production-like design than a generic Pandas agent.

In [107]:
response = nfl_agent_graph.invoke({
    "messages": [
        ("user", "Which were the 5 hignest scoring games in the dataset?")
    ]
})
print(response["messages"][-1].content)

<thinking>The tool has provided the top 10 highest scoring games. I will extract the top 5 from this list to answer the User's question.</thinking>

Here are the 5 highest scoring games in the dataset:

1. Week 4, 10/2/2022, DET vs SEA, Home Final Score: 45, Visitor Final Score: 48, Winner: SEA, Total Points: 93
2. Week 2, 9/18/2022, BAL vs MIA, Home Final Score: 38, Visitor Final Score: 42, Winner: MIA, Total Points: 80
3. Week 8, 10/30/2022, DAL vs CHI, Home Final Score: 49, Visitor Final Score: 29, Winner: DAL, Total Points: 78
4. Week 7, 10/20/2022, ARI vs NO, Home Final Score: 42, Visitor Final Score: 34, Winner: ARI, Total Points: 76
5. Week 1, 9/11/2022, DET vs PHI, Home Final Score: 35, Visitor Final Score: 38, Winner: PHI, Total Points: 73


In [109]:
response = nfl_agent_graph.invoke({
    "messages": [
        ("user", "Show me all available games for the Buffalo Bills.")
    ]
})

print(response["messages"][-1].content)

<thinking>The 'team_games_tool' has returned the requested information. I will now present the results to the User.</thinking>

Here are all the available games for the Buffalo Bills:

|   week | gameDate   | homeTeamAbbr   | visitorTeamAbbr   |   homeFinalScore |   visitorFinalScore | winner   |   scoreMargin |   totalPoints |
|-------:|:-----------|:---------------|:------------------|-----------------:|--------------------:|:---------|--------------:|--------------:|
|      1 | 9/8/2022   | LA             | BUF               |               10 |                  31 | BUF      |            21 |            41 |
|      2 | 9/19/2022  | BUF            | TEN               |               41 |                   7 | BUF      |            34 |            48 |
|      3 | 9/25/2022  | MIA            | BUF               |               21 |                  19 | MIA      |             2 |            40 |
|      4 | 10/2/2022  | BAL            | BUF               |               20 |           